In [1]:
!nvidia-smi

Wed Oct 21 04:44:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -q --force-reinstall --no-deps "pillow==11.3.0"

In [3]:
# Put pandas and pillow back to the versions Colab expects
!pip install -q "pandas==2.2.3" "pillow<13"
# Upgrade only what the model needs
!pip install -q -U "transformers>=4.57" accelerate

In [4]:
import os, re, json, math, time
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

# ---------------- SETTINGS ----------------
IMAGE_DIR = Path("/content")                 # images uploaded straight into /content
CSV_PATH  = "/content/cohort_ocr_v2.csv"
OUT_DIR   = "/content/qwen_ratings"

MODEL_ID  = "Qwen/Qwen3-VL-8B-Instruct"
PRECISION = "bf16"           # A100 40GB
MAX_PIXELS     = 2_000_000
MAX_OCR_CHARS  = 4000
MAX_NEW_TOKENS = 400
# ------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

# The six categories. This list order is also the tie-break order.
CATEGORIES = ["confirmshaming", "hidden_information", "preselection",
              "scarcity", "urgency", "social_proof"]

IMG_EXT = {".png", ".jpg", ".jpeg", ".webp"}
assert os.path.exists(CSV_PATH), "cohort_ocr_v2.csv is not in /content"
uploaded = sorted(p for p in IMAGE_DIR.rglob("*")
                  if p.is_file() and p.suffix.lower() in IMG_EXT and "sample_data" not in p.parts)
print(len(uploaded), "images uploaded:")
for p in uploaded:
    print("  ", p.name)

207 images uploaded:
   (170)shapermint.com_fcf2.jpg
   (177)www.adoreme.com_a719.jpg
   (194)www.dressbarn.com_30d1.jpg
   (23)nutrabay.com_5665.jpg
   (27)puffy.com_f607.jpg
   (35)shapermint.com_d46f.jpg
   (42)speedcubeshop.com_440a.jpg
   (662)www.bcbg.com_a8d3.jpg
   (69)www.crossrope.com_6d43.jpg
   (89)www.harveynorman.com.sg_bb02.jpg
   Arjun_01.png
   Arjun_02.png
   Arjun_03.png
   Arjun_04.png
   Arjun_05.png
   Arjun_06.png
   Arjun_07.png
   Arjun_08.png
   Arjun_09.png
   Arjun_10.png
   Arjun_11_Negative.png
   Arjun_12_Negative.png
   Arjun_13_Negative.png
   Arjun_14_Negative.png
   Arjun_15_Negative(1).png
   Arjun_15_Negative.png
   Arjun_16_Negative.png
   Arjun_17_Hidden.png
   Arjun_18_Negative.png
   Arjun_19_Negative.png
   Arjun_20_Negative.png
   Arjun_21_Negative.png
   Arjun_22.jpg
   Arjun_23.jpg
   Arjun_24.jpg
   Arjun_25.jpg
   Arjun_26.jpg
   Arjun_27.jpg
   Arjun_28.jpg
   Arjun_29.jpg
   Arjun_30.jpg
   Arjun_31.jpg
   Arjun_32.jpg
   Arjun_33.jpg
  

In [5]:
df = pd.read_csv(CSV_PATH, keep_default_na=False)
for c in ["image", "collector", "label"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

print(df.shape)
print(df.columns.tolist())

IMAGE_COL = "image"
LABEL_COL = "label"

# Find the OCR text column
OCR_COL = None
for c in df.columns:
    if c.strip().lower() in {"ocr", "ocr_text", "ocr_output", "ocr_v2", "ocr_result", "text", "extracted_text"}:
        OCR_COL = c
        break
if OCR_COL is None:
    ocr_like = [c for c in df.columns if "ocr" in c.lower()]
    OCR_COL = ocr_like[0] if ocr_like else None

print("IMAGE_COL =", IMAGE_COL, "| LABEL_COL =", LABEL_COL, "| OCR_COL =", OCR_COL)
df.head(3)

(180, 8)
['image', 'collector', 'platform', 'label', 'all_labels', 'justification', 'text', 'text_old']
IMAGE_COL = image | LABEL_COL = label | OCR_COL = text


,image,collector,platform,label,all_labels,justification,text,text_old
0,Arjun_01.png,Arjun,web,preselection,preselection,Pop up to add items to your order that You wer...,Back to store Complete your order Delivery det...,x Complete your order Dr Pepper $2.70 Popular ...
1,Arjun_02.png,Arjun,web,hidden_information,hidden_information;preselection,Promises a price and discount throughout the w...,Deiveiy Oplons Send asa gift Priority Add a pe...,Meivery Veils i Send asa gift > ae Add a perso...
2,Arjun_03.png,Arjun,web,urgency,urgency,Coundown timer on a sale that was there yester...,Free Shipping by Amazon Get FREE Shipping on e...,{-) Free Shipping by Amazon Get FREE Shipping ...


In [6]:
# OCR_COL = "..."   # uncomment and set it if Cell 4 printed None or the wrong column
assert OCR_COL, "Set OCR_COL to the column that holds the OCR text"

def image_identity(name):
    """File name without extension, capitals, or a leading '(170)' number."""
    p = Path(name)
    stem = p.stem if p.suffix.lower() in IMG_EXT else p.name
    return re.sub(r"^\(\d+\)\s*", "", stem).casefold()

def norm_label(x):
    return str(x).strip().lower().replace("-", "_").replace(" ", "_")

def find_image(name):
    m = [p for p in uploaded if p.name == name] or \
        [p for p in uploaded if image_identity(p.name) == image_identity(name)]
    return str(m[0]) if len(m) == 1 else None

df["image_path"] = df[IMAGE_COL].map(find_image)
df["label_norm"] = df[LABEL_COL].map(norm_label)

matched = df[df["image_path"].notna()]
print(f"{len(matched)} of {len(uploaded)} uploaded images matched a row in the CSV\n")
print(matched[[IMAGE_COL, LABEL_COL]].to_string(index=False))

unmatched = [p.name for p in uploaded if str(p) not in set(matched["image_path"])]
if unmatched:
    print("\nNot found in the CSV:", unmatched)

180 of 207 uploaded images matched a row in the CSV

                image              label
         Arjun_01.png       preselection
         Arjun_02.png hidden_information
         Arjun_03.png            urgency
         Arjun_04.png            urgency
         Arjun_05.png            urgency
         Arjun_06.png            urgency
         Arjun_07.png       social_proof
         Arjun_08.png       social_proof
         Arjun_09.png            urgency
         Arjun_10.png           scarcity
  Arjun_17_Hidden.png hidden_information
        Khushi_01.png       social_proof
        Khushi_02.png     confirmshaming
        Khushi_03.png            urgency
        Khushi_04.png     confirmshaming
        Khushi_05.png            urgency
        Khushi_06.png           scarcity
        Khushi_07.png       social_proof
        Khushi_08.png       preselection
        Khushi_10.png hidden_information
        Khushi_11.png           scarcity
        Khushi_12.png hidden_information
    

In [7]:
SYSTEM_PROMPT = """You are auditing one app or website screenshot for dark patterns.

You receive two inputs: the screenshot and OCR text extracted from it. The screenshot is the ground truth. OCR can miss small or faint text, merge lines, or misread words, so use it to help you read, but confirm every cue against the image. Whether a box is checked, a toggle is on, or text is tiny or low contrast can only be judged from the image.

Use only visible evidence. Do not speculate about content below the viewport, later interactions, the factual truth of offers, or hidden behavior.

Rate EACH of the six categories independently, using these evidence tests.

confirmshaming:
The decline button or link uses guilt, shame, negative identity or a loss-framed first-person refusal to pressure acceptance. This includes framing refusal as preferring to lose a benefit, spend unnecessarily, or lack a desirable identity. Inspect even small or low-contrast refusal text in offer and signup overlays. A bare decline is not enough: identify the evaluative or loss-loaded wording. Neutral Skip, No thanks, or Continue Without Offer does not qualify.
Hints: 2 = a clearly loss-framed or guilt-laden decline such as 'No thanks, I prefer paying full price'. 3 = the shaming decline is prominent or openly insults the user, such as 'No, I don't care about my health'.

hidden_information:
A material renewal charge, mandatory fee, contractual condition or meaningful alternative is visibly difficult to notice because it is tiny, low contrast or suppressed. In pricing tables, a materially higher renewal price or recurring charge printed much smaller or lower contrast than a large introductory price qualifies even if the text is readable. Compare the visual prominence of the introductory price and the future charge. Normal scrolling, navigation, descriptions, decorative labels or a high price alone are not hidden information.
Hints: 2 = a renewal price or fee clearly smaller or fainter than the headline price. 3 = a material charge or condition that is nearly unreadable next to a large promotional price.

preselection:
An OPTIONAL seller-favoring choice is visibly already checked, toggled on, or selected by default, so the user must undo it. This includes paid add-ons, selected subscriptions, marketing consent and non-essential tracking. A visible checked box or enabled toggle establishes its current selected state; the choice need not be paid. Interpret control polarity: if a control disables, blocks or opts out of an optional seller-benefiting activity, its OFF state can leave that underlying activity enabled and therefore qualify as preselection. An OFF opt-out control is different from an OFF optional add-on. Necessary security or system settings do not qualify. A recommendation, Popular badge, colored card, upsell popup or plus button does not establish a selected choice.
Hints: 2 = one optional box or toggle (for example marketing emails) already on. 3 = a paid add-on or subscription preselected, or several optional choices preselected.

scarcity:
Explicit remaining-stock or limited-quantity language such as only one left, almost sold out, or people holding stock in their carts. A discount, timer, review count, purchase count, bestseller badge or high price is not scarcity.
Hints: 2 = a plain stock message such as 'Only 3 left'. 3 = prominent, highlighted or repeated stock warnings.

urgency:
An explicit countdown or deadline for an offer or action. Delivery-duration estimates are not countdowns. Limited time is urgency, not limited quantity.
Hints: 2 = a stated deadline such as 'Offer ends today' or 'Limited time'. 3 = a visible running countdown timer or a prominent deadline banner.

social_proof:
Visible purchase counts or claims about other people's purchases, views or activity used as persuasion. Ordinary star ratings or reviews alone do not qualify. Sold counts can qualify.
Hints: 2 = a count such as '1.2k sold' or '25 people bought this'. 3 = prominent live-activity claims such as '12 people are viewing this right now'.

Rating scale, applied to each category on its own:
0 = no sign. The evidence test is not met. Anything the test says does not qualify is a 0.
1 = weak sign. A cue that might meet the test but is ambiguous, partly legible, or borderline.
2 = clear sign. The evidence test is clearly met.
3 = strong, obvious sign. The test is clearly met and the cue is prominent, emphasized, or repeated.
Several categories can be above 0 at once. Most screens have most categories at 0. Never raise a rating without a visible cue.

Output format. Return ONLY one valid JSON object, with no markdown and no text before or after it, in exactly this shape:
{"justification": "...", "ratings": {"confirmshaming": 0, "hidden_information": 0, "preselection": 0, "scarcity": 0, "urgency": 0, "social_proof": 0}}

justification: one or two sentences. Name the visible cue behind every rating above 0 and quote its on-screen words in single quotes, for example: Decline link reads 'No, I like paying more'. If every rating is 0, say briefly why nothing qualifies. Never use double quotes inside the justification.
ratings: all six keys, each an integer 0, 1, 2 or 3.
Do not add a primary label or any other key."""

FIX_PROMPT = ("Your reply was not valid ({err}). Reply again with ONLY the JSON object: "
              "keys 'justification' (string) and 'ratings' (all six categories, integers 0-3). No other text.")
print(len(SYSTEM_PROMPT), "characters")

5238 characters


In [8]:
def parse_output(text):
    """Return (result, None) if the reply is valid, else (None, reason)."""
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    m = re.search(r"\{.*\}", t, re.S)
    if not m:
        return None, "no JSON object found"
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError as e:
        return None, f"bad JSON: {e}"
    if not isinstance(obj, dict) or set(obj) != {"justification", "ratings"}:
        return None, f"top-level keys must be justification and ratings, got {list(obj) if isinstance(obj, dict) else type(obj).__name__}"
    just, r = obj["justification"], obj["ratings"]
    if not isinstance(just, str) or not just.strip():
        return None, "justification must be a non-empty string"
    if not isinstance(r, dict) or set(r) != set(CATEGORIES):
        return None, f"ratings keys must be exactly {CATEGORIES}"
    for c in CATEGORIES:
        v = r[c]
        if isinstance(v, bool) or not isinstance(v, int) or not 0 <= v <= 3:
            return None, f"rating for {c} must be an integer 0-3, got {v!r}"
    return {"justification": just.strip(), "ratings": {c: r[c] for c in CATEGORIES}}, None


def predict(ratings):
    """Highest-rated category; ties go to the earlier category in CATEGORIES.
    All zeros -> 'none'. Strength = the highest rating."""
    top = max(ratings[c] for c in CATEGORIES)
    if top == 0:
        return "none", 0
    for c in CATEGORIES:
        if ratings[c] == top:
            return c, top

# quick self-check
assert predict({c: 0 for c in CATEGORIES}) == ("none", 0)
assert predict({**{c: 0 for c in CATEGORIES}, "urgency": 2, "preselection": 2}) == ("preselection", 2)
assert predict({**{c: 0 for c in CATEGORIES}, "social_proof": 3, "confirmshaming": 1}) == ("social_proof", 3)
print("parser and labeling rule OK")

parser and labeling rule OK


In [9]:
from transformers import AutoProcessor, AutoModelForImageTextToText

DTYPE = {"fp32": torch.float32, "bf16": torch.bfloat16}[PRECISION]

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map="auto",
    attn_implementation="sdpa",
).eval()

total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU:", torch.cuda.get_device_name(0), "| %.0f GB" % total_gb)
print("dtype:", model.dtype, "| GPU memory used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

GPU: NVIDIA A100-SXM4-40GB | 42 GB
dtype: torch.bfloat16 | GPU memory used: 17.5 GB


In [13]:
def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    if w * h > MAX_PIXELS:
        s = math.sqrt(MAX_PIXELS / (w * h))
        img = img.resize((max(1, int(w * s)), max(1, int(h * s))), Image.LANCZOS)
    return img


def build_messages(row):
    ocr = row[OCR_COL]
    ocr = "" if pd.isna(ocr) else str(ocr).strip()
    if len(ocr) > MAX_OCR_CHARS:
        ocr = ocr[:MAX_OCR_CHARS] + " ...[truncated]"
    user_text = (
        "OCR text extracted from this screenshot (may contain errors or miss small text):\n"
        f"<<<\n{ocr or '(no OCR text)'}\n>>>\n\n"
        "Rate the screenshot now. Return only the JSON object."
    )
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]},
    ]


@torch.inference_mode()
def generate(messages, image):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


def error_result(row, err):
    return {"image": str(row[IMAGE_COL]), "label": str(row[LABEL_COL]), "raw": "",
            "justification": None, "ratings": None, "error": err, "attempts": 0}


def run_one(row, retries=1):
    img = load_image(row["image_path"])
    messages = build_messages(row)
    raw = generate(messages, img)
    parsed, err = parse_output(raw)
    attempts = 1
    while err and attempts <= retries:
        messages = messages + [
            {"role": "assistant", "content": [{"type": "text", "text": raw}]},
            {"role": "user", "content": [{"type": "text", "text": FIX_PROMPT.format(err=err)}]},
        ]
        raw = generate(messages, img)
        parsed, err = parse_output(raw)
        attempts += 1
    return {
        "image": str(row[IMAGE_COL]), "label": str(row[LABEL_COL]), "raw": raw,
        "justification": parsed["justification"] if parsed else None,
        "ratings": parsed["ratings"] if parsed else None,
        "error": err, "attempts": attempts,
    }

In [14]:
TEST_SET = "contextdp"   # "contextdp" = the 10 (number)site.com screenshots
                         # "cohort"    = 10 screenshots from the 180, two per label

if TEST_SET == "contextdp":
    ctx = sorted(p for p in uploaded if re.match(r"^\(\d+\)", p.name))
    test_df = pd.DataFrame({
        IMAGE_COL: [p.name for p in ctx],
        LABEL_COL: "(no label)",
        OCR_COL: "",                       # no OCR text for these in cohort_ocr_v2.csv
        "image_path": [str(p) for p in ctx],
    })
else:
    test_df = df[df["image_path"].notna()].sample(frac=1, random_state=0).groupby("label_norm").head(2).head(10)

print(f"Testing on {len(test_df)} screenshots ({TEST_SET})")

test_results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    test_results.append(run_one(row, retries=0))

valid = sum(r["error"] is None for r in test_results)
print(f"\nValid JSON on first try: {valid}/{len(test_results)}\n")
for r in test_results:
    print("=" * 90)
    print(r["image"], "| label:", r["label"])
    if r["error"]:
        print("INVALID:", r["error"])
        print("RAW:", r["raw"])
    else:
        pred, strength = predict(r["ratings"])
        print("predicted:", pred, "| strength:", strength)
        print("ratings:", r["ratings"])
        print("justification:", r["justification"])

Testing on 10 screenshots (contextdp)


  0%|          | 0/10 [00:00<?, ?it/s]


Valid JSON on first try: 10/10

(170)shapermint.com_fcf2.jpg | label: (no label)
predicted: urgency | strength: 2
ratings: {'confirmshaming': 0, 'hidden_information': 0, 'preselection': 0, 'scarcity': 0, 'urgency': 2, 'social_proof': 0}
justification: The sale ends in a countdown timer, which is a visible deadline. The text 'Hurry! This special sale ends in' indicates urgency. No other dark patterns are visible.
(177)www.adoreme.com_a719.jpg | label: (no label)
predicted: none | strength: 0
ratings: {'confirmshaming': 0, 'hidden_information': 0, 'preselection': 0, 'scarcity': 0, 'urgency': 0, 'social_proof': 0}
justification: No dark pattern cues are visible. There is no decline text with guilt or shame, no hidden fees, no preselected options, no scarcity messaging, no urgency countdown beyond the timer for a membership offer, and no social proof counts.
(194)www.dressbarn.com_30d1.jpg | label: (no label)
predicted: confirmshaming | strength: 2
ratings: {'confirmshaming': 2, 'hidden_i

In [15]:
RESULTS_JSONL = f"{OUT_DIR}/raw_outputs.jsonl"

done = set()
if os.path.exists(RESULTS_JSONL):
    with open(RESULTS_JSONL) as f:
        for line in f:
            r = json.loads(line)
            if r["error"] is None:
                done.add(r["image"])
print("Already done:", len(done))

has_image = df["image_path"].notna()
print(f"Screenshots with an uploaded image: {has_image.sum()}/{len(df)}")
todo = df[has_image & ~df[IMAGE_COL].astype(str).isin(done)]

start = time.time()
with open(RESULTS_JSONL, "a") as f:
    for _, row in tqdm(todo.iterrows(), total=len(todo)):
        try:
            res = run_one(row, retries=1)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            res = error_result(row, "CUDA out of memory (lower MAX_PIXELS and rerun)")
        except Exception as e:
            res = error_result(row, f"{type(e).__name__}: {e}")
        f.write(json.dumps(res) + "\n")
        f.flush()

print(f"Finished {len(todo)} screenshots in {(time.time() - start) / 60:.1f} min")

Already done: 0
Screenshots with an uploaded image: 180/180


  0%|          | 0/180 [00:00<?, ?it/s]

Finished 180 screenshots in 15.8 min


In [16]:
latest = {}
with open(RESULTS_JSONL) as f:
    for line in f:
        r = json.loads(line)
        if r["image"] not in latest or r["error"] is None:   # keep a valid result over an error
            latest[r["image"]] = r

records = []
for _, row in df.iterrows():
    r = latest.get(str(row[IMAGE_COL]))
    rec = {"image": row[IMAGE_COL], "label": row[LABEL_COL]}
    if r and r["error"] is None:
        pred, strength = predict(r["ratings"])
        rec.update(predicted=pred, justification=r["justification"], **r["ratings"], strength=strength)
    else:
        reason = r["error"] if r else "not run"
        rec.update(predicted="ERROR", justification=f"ERROR: {reason}",
                   **{c: None for c in CATEGORIES}, strength=None)
    records.append(rec)

ratings = pd.DataFrame(records, columns=["image", "label", "predicted", "justification", *CATEGORIES, "strength"])
ratings[CATEGORIES + ["strength"]] = ratings[CATEGORIES + ["strength"]].astype("Int64")

out_path = f"{OUT_DIR}/ratings.csv"
ratings.to_csv(out_path, index=False)
print("Saved", out_path, "|", len(ratings), "rows |", (ratings["predicted"] == "ERROR").sum(), "errors")
ratings.head(10)

Saved /content/qwen_ratings/ratings.csv | 180 rows | 0 errors


,image,label,predicted,justification,confirmshaming,hidden_information,preselection,scarcity,urgency,social_proof,strength
0,Arjun_01.png,preselection,none,No dark pattern cues are visible. The 'Skip' b...,0,0,0,0,0,0,0
1,Arjun_02.png,hidden_information,none,No dark pattern cues are visible. There is no ...,0,0,0,0,0,0,0
2,Arjun_03.png,urgency,none,No dark pattern cues are visible. No decline t...,0,0,0,0,0,0,0
3,Arjun_04.png,urgency,hidden_information,The offer includes a visible countdown 'Offer ...,0,2,0,0,2,0,2
4,Arjun_05.png,urgency,urgency,The offer has a visible countdown timer showin...,0,0,0,0,2,0,2
5,Arjun_06.png,urgency,none,No dark pattern cues are visible. The offer ha...,0,0,0,0,0,0,0
6,Arjun_07.png,social_proof,none,No dark pattern cues are visible. No decline t...,0,0,0,0,0,0,0
7,Arjun_08.png,social_proof,none,No dark pattern cues are visible. There is no ...,0,0,0,0,0,0,0
8,Arjun_09.png,urgency,urgency,The countdown timer showing '9 DAYS : 7 HOURS ...,0,0,0,0,2,0,2
9,Arjun_10.png,scarcity,scarcity,The 'Only 6 left in stock - order soon.' text ...,0,0,0,2,0,2,2


In [17]:
ok = ratings["predicted"] != "ERROR"
gold = ratings["label"].map(norm_label)
print(f"Exact match with label: {(gold[ok] == ratings.loc[ok, 'predicted']).mean():.1%} on {ok.sum()} screenshots")
display(pd.crosstab(gold[ok], ratings.loc[ok, "predicted"], rownames=["label"], colnames=["predicted"]))

from google.colab import files
files.download(out_path)

Exact match with label: 63.3% on 180 screenshots


predicted,confirmshaming,hidden_information,none,preselection,scarcity,social_proof,urgency
label,,,,,,,
confirmshaming,35,0,10,0,0,1,0
hidden_information,0,7,19,0,0,0,0
none,0,0,54,0,0,0,1
preselection,0,2,12,3,0,0,1
scarcity,0,0,0,0,5,0,1
social_proof,0,0,12,0,1,1,0
urgency,0,1,5,0,0,0,9


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>